In [1]:
import pandas as pd
import joblib
import time
from glob import glob
import numpy as np
import statistics as st
import matplotlib.pyplot as plt
import ee
import ast

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
acz_list = [
            'Western Himalayan Region',
            # 'Eastern Himalayan Region',
            # 'Lower Gangetic Plain Region',
            # 'Middle Gangetic Plain Region',
            # 'Upper Gangetic Plain Region',
            # 'Trans Gangetic Plain Region',
            # 'Eastern Plateau & Hills Region',
            # 'Central Plateau & Hills Region',
            # 'Western Plateau and Hills Region',
            # 'Southern Plateau and Hills Region',
            # 'East Coast Plains & Hills Region'
            ]

In [4]:
best_month_dict = {'Eastern Plateau & Hills Region': 'cc_12',
                   'Middle Gangetic Plain Region': 'cc_10',
                   'Lower Gangetic Plain Region': 'cc_9',
                   'Western Himalayan Region': 'cc_8',
                   'Eastern Himalayan Region': 'cc_10',
                   'Upper Gangetic Plain Region': 'cc_9',
                   'Trans Gangetic Plain Region': 'cc_9',
                   'Central Plateau & Hills Region': 'cc_7',
                   'Western Plateau and Hills Region': 'cc_11',
                   'Southern Plateau and Hills Region': 'cc_8',
                   'East Coast Plains & Hills Region': 'cc_12'}

In [18]:
# if AY 2023's data is being added, set year = '2023'
# No correction for 2016, starts from 2017
# (For correcting 2017, set to year to 2019)
year = '2024'

# We need to correct data from year_2 and year_1
year_1 = int(year)-1
year_2 = int(year)-2
year_3 = int(year)-3
year_4 = int(year)-4

In [7]:
# df = pd.read_csv(f'drive/MyDrive/TreeHealth/Agroclimatic_regions/{acz_list[0]}.csv')
# dist_list = list(df['Name'])
# dist_list = ['Katihar', 'Kishanganj']
# print(len(dist_list))
# print(f'dist_list: {dist_list}')

2
dist_list: ['Katihar', 'Kishanganj']


# Data Correction - CCD

In [ ]:
def ccd_corrections_2017(df):
     columns = list(df.columns)
     n = len(columns)

     # In case the no. of columns are too less for any corrections to be performed
     if n < 5:
         print(f"Number of columns are {n}, which is too less to perform any corrections..")
         correction_df = pd.DataFrame(columns=columns)
         return correction_df

     # Correcting the second (2017) year
     correction_df = df[(df[columns[n-1]] == df[columns[n-2]]) & (df[columns[n-2]] == df[columns[n-4]]) & (df[columns[n-3]] !=df[columns[n-1]])]
     correction_df.drop_duplicates(inplace=True)

     # Actually Performing the corrections once all rows where corrections need to be performed are found
     correction_df.loc[(correction_df[columns[n-1]] == correction_df[columns[n-2]]) &
      (correction_df[columns[n-2]] == correction_df[columns[n-4]]) &
       (correction_df[columns[n-3]] != correction_df[columns[n-1]]), columns[n-3]] = correction_df[columns[n-1]]

     return correction_df


def corrections(df):
    columns = list(df.columns)
    n = len(columns)

    # In case the no. of columns are too less for any corrections to be performed
    if n < 5:
        correction_df = pd.DataFrame(columns=columns)
        return correction_df

    # Correcting the second last year
    correction_df = df[(df[columns[n-1]] == df[columns[n-3]]) & (df[columns[n-3]] == df[columns[n-4]]) & (df[columns[n-2]] != df[columns[n-1]])]
    correction_df.drop_duplicates(inplace=True)
    print("correction_df 1", correction_df)
    # Correcting the middle year
    new_df = df[(df[columns[n-5]] == df[columns[n-4]]) & (df[columns[n-4]] == df[columns[n-2]]) & (df[columns[n-2]] == df[columns[n-1]]) & (df[columns[n-3]] != df[columns[n-5]])]
    correction_df = pd.concat([correction_df, new_df], ignore_index=True)
    correction_df.drop_duplicates(inplace=True)
    del(new_df)
    print("correction_df 2", correction_df)
    # Actually Performing the corrections once all rows where corrections need to be performed are found
    correction_df.loc[(correction_df[columns[n-1]] == correction_df[columns[n-3]]) & (correction_df[columns[n-3]] == correction_df[columns[n-4]]) &
     (correction_df[columns[n-2]] != correction_df[columns[n-1]]), columns[n-2]] = correction_df[columns[n-1]]
    correction_df.loc[(correction_df[columns[n-5]] == correction_df[columns[n-4]]) & (correction_df[columns[n-4]] == correction_df[columns[n-2]]) &
     (correction_df[columns[n-2]] == correction_df[columns[n-1]]) & (correction_df[columns[n-3]] != correction_df[columns[n-5]]), columns[n-3]] = correction_df[columns[n-5]]

    return correction_df

In [ ]:
for agroclimatic_zone in acz_list:
    print(agroclimatic_zone)
    df = pd.read_csv(f'drive/MyDrive/TreeHealth/Agroclimatic_regions/{agroclimatic_zone}.csv')
    dist_list = list(df['Name'])
    print(f'len(dist_list): {len(dist_list)}')

    path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/'

    total_corrections = 0
    total_length = 0

    for i in range(len(dist_list)):

        # if i != 11:
        #   continue

        print(i, dist_list[i])
        file_4 = path + dist_list[i] + f"/{year_4}/result_monthly_cc.csv"
        file_3 = path + dist_list[i] + f"/{year_3}/result_monthly_cc.csv"
        file_2 = path + dist_list[i] + f"/{year_2}/result_monthly_cc.csv"
        file_1 = path + dist_list[i] + f"/{year_1}/result_monthly_cc.csv"
        file_0 = path + dist_list[i] + f"/{year}/result_monthly_cc.csv"

        band = best_month_dict[agroclimatic_zone]
        columnList = ['cc_1', 'cc_2', 'cc_3', 'cc_4', 'cc_5', 'cc_6', 'cc_7', 'cc_8', 'cc_9', 'cc_10', 'cc_11', 'cc_12']
        columnList.remove(band)

        try:
            df_4 = pd.read_csv(file_4)
            df_4.drop(columns=columnList, inplace=True)
            df_4.rename(columns={band: 'cover_class'}, inplace=True)

        except Exception as e:
            print(e)
            df_4 = pd.DataFrame(columns=['.geo', 'cover_class'])

        try:
            df_3 = pd.read_csv(file_3)
            df_3.drop(columns=columnList, inplace=True)
            df_3.rename(columns={band: 'cover_class'}, inplace=True)

        except Exception as e:
            print(e)
            df_3 = pd.DataFrame(columns=['.geo', 'cover_class'])

        df_4.rename(columns={'cover_class': f'cc_{year_4}'}, inplace=True)
        df_3.rename(columns={'cover_class': f'cc_{year_3}'}, inplace=True)
        merged_df = pd.merge(df_4, df_3, on='.geo', how='outer')
        del(df_4)
        del(df_3)

        try:
            df_2 = pd.read_csv(file_2)
            df_2.drop(columns=columnList, inplace=True)
            df_2.rename(columns={band: 'cover_class'}, inplace=True)

        except Exception as e:
            print(e)
            df_2 = pd.DataFrame(columns=['.geo', 'cover_class'])

        df_2.rename(columns={'cover_class': f'cc_{year_2}'}, inplace=True)
        merged_df = pd.merge(merged_df, df_2, on='.geo', how='outer')
        del(df_2)

        try:
            df_1 = pd.read_csv(file_1)
            df_1.drop(columns=columnList, inplace=True)
            df_1.rename(columns={band: 'cover_class'}, inplace=True)
        except Exception as e:
            print(e)
            df_1 = pd.DataFrame(columns=['.geo', 'cover_class'])

        df_1.rename(columns={'cover_class': f'cc_{year_1}'}, inplace=True)
        merged_df = pd.merge(merged_df, df_1, on='.geo', how='outer')
        del(df_1)

        try:
            df_0 = pd.read_csv(file_0)
            df_0.drop(columns=columnList, inplace=True)
            df_0.rename(columns={band: 'cover_class'}, inplace=True)

        except Exception as e:
            print(e)
            df_0 = pd.DataFrame(columns=['.geo', 'cover_class'])

        df_0.rename(columns={'cover_class': f'cc_{year}'}, inplace=True)
        merged_df = pd.merge(merged_df, df_0, on='.geo', how='outer')
        del(df_0)


        merged_df = merged_df[['.geo', f'cc_{year_4}', f'cc_{year_3}', f'cc_{year_2}', f'cc_{year_1}', f'cc_{year}']]
        print(merged_df)
        total_length += len(merged_df)
        print("Length of merged_df:", len(merged_df))
        print("Total Length:", total_length)
        if year == '2019':
          print("Correcting 2017")
          correction_df = ccd_corrections_2017(merged_df)
        else:
          correction_df = corrections(merged_df)

        del(merged_df)
        total_corrections += len(correction_df)
        print(f'Correction_df length: {len(correction_df)}')
        print(f'Total Corrections: {total_corrections}')
        if len(correction_df) > 0:
            # if year != last_year:
            correction_year_2 = correction_df[['.geo', f'cc_{year_2}']]
            correction_year_2.to_csv(f'{path}{dist_list[i]}/{year_2}/result_monthly_cc_corrections.csv', index=False)
            if year != '2019':
              correction_year_1 = correction_df[['.geo', f'cc_{year_1}']]
              correction_year_1.to_csv(f'{path}{dist_list[i]}/{year_1}/result_monthly_cc_corrections.csv', index=False) # Comment when correcting 2017

        del(correction_df)

East Coast Plains & Hills Region
len(dist_list): 68
0 Chittoor


KeyboardInterrupt: 

 # Data Correction - CH

In [6]:
# Column Order
# ['.geo',
# 'rh98_{year_4}', 'rh98_{year_3}', 'rh98_{year_2}', 'rh98_{year_1}', 'rh98_{year}',
# 'rh75_{year_4}', 'rh75_{year_3}', 'rh75_{year_2}', 'rh75_{year_1}', 'rh75_{year}',
# 'rh50_{year_4}', 'rh50_{year_3}', 'rh50_{year_2}', 'rh50_{year_1}', 'rh50_{year}']


def ch_corrections(df):
    columns = list(df.columns)
    n = len(columns)

    # Correcting the second last year - rh98
    correction_df = df[(df[columns[5]] == df[columns[3]]) & (df[columns[3]] == df[columns[2]]) & (df[columns[4]] != df[columns[5]])]
    correction_df.drop_duplicates(inplace=True)

    # Correcting the middle year - rh98
    new_df = df[(df[columns[1]] == df[columns[2]]) & (df[columns[2]] == df[columns[4]]) & (df[columns[4]] == df[columns[5]]) & (df[columns[3]] != df[columns[1]])]
    correction_df = pd.concat([correction_df, new_df], ignore_index=True)
    correction_df.drop_duplicates(inplace=True)
    del(new_df)


    # Correcting the second last year - rh75
    new_df = df[(df[columns[10]] == df[columns[8]]) & (df[columns[8]] == df[columns[7]]) & (df[columns[9]] != df[columns[10]])]
    correction_df = pd.concat([correction_df, new_df], ignore_index=True)
    del(new_df)
    correction_df.drop_duplicates(inplace=True)

    # Correcting the middle year - rh75
    new_df = df[(df[columns[6]] == df[columns[7]]) & (df[columns[7]] == df[columns[9]]) & (df[columns[9]] == df[columns[10]]) & (df[columns[8]] != df[columns[6]])]
    correction_df = pd.concat([correction_df, new_df], ignore_index=True)
    correction_df.drop_duplicates(inplace=True)
    del(new_df)


    # Correcting the second last year - rh50
    new_df = df[(df[columns[15]] == df[columns[13]]) & (df[columns[13]] == df[columns[12]]) & (df[columns[14]] != df[columns[15]])]
    correction_df = pd.concat([correction_df, new_df], ignore_index=True)
    del(new_df)
    correction_df.drop_duplicates(inplace=True)

    # Correcting the middle year - rh50
    new_df = df[(df[columns[11]] == df[columns[12]]) & (df[columns[12]] == df[columns[14]]) & (df[columns[14]] == df[columns[15]]) & (df[columns[13]] != df[columns[11]])]
    correction_df = pd.concat([correction_df, new_df], ignore_index=True)
    correction_df.drop_duplicates(inplace=True)
    del(new_df)


    # Actually Performing the corrections once all rows where corrections need to be performed are found

    # rh98
    correction_df.loc[(correction_df[columns[5]] == correction_df[columns[3]]) & (correction_df[columns[3]] == correction_df[columns[2]]) &
     (correction_df[columns[4]] != correction_df[columns[5]]), columns[4]] = correction_df[columns[5]]
    correction_df.loc[(correction_df[columns[1]] == correction_df[columns[2]]) & (correction_df[columns[2]] == correction_df[columns[4]]) &
     (correction_df[columns[4]] == correction_df[columns[5]]) & (correction_df[columns[3]] != correction_df[columns[1]]), columns[3]] = correction_df[columns[1]]


    # rh75
    correction_df.loc[(correction_df[columns[10]] == correction_df[columns[8]]) & (correction_df[columns[8]] == correction_df[columns[7]]) &
     (correction_df[columns[9]] != correction_df[columns[10]]), columns[9]] = correction_df[columns[10]]
    correction_df.loc[(correction_df[columns[6]] == correction_df[columns[7]]) & (correction_df[columns[7]] == correction_df[columns[9]]) &
     (correction_df[columns[9]] == correction_df[columns[10]]) & (correction_df[columns[8]] != correction_df[columns[6]]), columns[8]] = correction_df[columns[6]]


    # rh50
    correction_df.loc[(correction_df[columns[15]] == correction_df[columns[13]]) & (correction_df[columns[13]] == correction_df[columns[12]]) &
     (correction_df[columns[14]] != correction_df[columns[15]]), columns[14]] = correction_df[columns[15]]
    correction_df.loc[(correction_df[columns[11]] == correction_df[columns[12]]) & (correction_df[columns[12]] == correction_df[columns[14]]) &
     (correction_df[columns[14]] == correction_df[columns[15]]) & (correction_df[columns[13]] != correction_df[columns[11]]), columns[13]] = correction_df[columns[11]]

    return correction_df

def ch_corrections_2017(df):
     columns = list(df.columns)
     n = len(columns)

     # Correcting the second last year - rh98
     correction_df = df[(df[columns[5]] == df[columns[4]]) & (df[columns[4]] == df[columns[2]]) & (df[columns[3]] != df[columns[5]])]
     correction_df.drop_duplicates(inplace=True)

     # Correcting the second last year - rh75
     new_df = df[(df[columns[10]] == df[columns[9]]) & (df[columns[9]] == df[columns[7]]) & (df[columns[8]] != df[columns[10]])]
     correction_df = pd.concat([correction_df, new_df], ignore_index=True)
     del(new_df)
     correction_df.drop_duplicates(inplace=True)

     # Correcting the second last year - rh50
     new_df = df[(df[columns[15]] == df[columns[14]]) & (df[columns[14]] == df[columns[12]]) & (df[columns[13]] != df[columns[15]])]
     correction_df = pd.concat([correction_df, new_df], ignore_index=True)
     del(new_df)
     correction_df.drop_duplicates(inplace=True)

     # Actually Performing the corrections once all rows where corrections need to be performed are found

     # rh98
     correction_df.loc[(correction_df[columns[5]] == correction_df[columns[4]]) & (correction_df[columns[4]] == correction_df[columns[2]]) &
      (correction_df[columns[3]] != correction_df[columns[5]]), columns[3]] = correction_df[columns[5]]

     # rh75
     correction_df.loc[(correction_df[columns[10]] == correction_df[columns[9]]) & (correction_df[columns[9]] == correction_df[columns[7]]) &
      (correction_df[columns[8]] != correction_df[columns[10]]), columns[8]] = correction_df[columns[10]]

     # rh50
     correction_df.loc[(correction_df[columns[15]] == correction_df[columns[14]]) & (correction_df[columns[14]] == correction_df[columns[12]]) &
      (correction_df[columns[13]] != correction_df[columns[15]]), columns[13]] = correction_df[columns[15]]

     return correction_df

CH corrections without Chunking

In [19]:
# Function to convert string representation of list to an actual list
def convert_to_list(string):
    return ast.literal_eval(string)


for agroclimatic_zone in acz_list:
    print(agroclimatic_zone)

    df = pd.read_csv('drive/MyDrive/TreeHealth/district_to_agroclimaticZone_mapping.csv')
    df['IntersectingZones'] = df['IntersectingZones'].apply(convert_to_list)
    district_mapping_df = df[df['AgroclimaticZone'] == agroclimatic_zone][['District', 'IntersectingZones']]
    dist_list = []
    for ind in district_mapping_df.index:
        district = district_mapping_df.loc[ind, 'District']
        zones = district_mapping_df['IntersectingZones'][ind]
        dist_list.append(district)

    dist_list = ['Dehradun', 'Garhwal', 'Nainital', 'Champawat']
    print(f'len(dist_list): {len(dist_list)}')
    print(f'dist_list=: {dist_list}')

    path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/'

    total_corrections = 0
    total_length = 0

    for i in range(len(dist_list)):

        if i == 4 or i==6:
          continue

        print(i, dist_list[i])
        file_4 = path + dist_list[i] + f"/{year_4}/result_chm.csv"
        file_3 = path + dist_list[i] + f"/{year_3}/result_chm.csv"
        file_2 = path + dist_list[i] + f"/{year_2}/result_chm.csv"
        file_1 = path + dist_list[i] + f"/{year_1}/result_chm.csv"
        file_0 = path + dist_list[i] + f"/{year}/result_chm.csv"

        try:
            df_4 = pd.read_csv(file_4)
            df_4.rename(columns={'rh98_class': f'rh98_{year_4}', 'rh75_class': f'rh75_{year_4}', 'rh50_class': f'rh50_{year_4}'}, inplace=True)

        except Exception as e:
            print(e)
            df_4 = pd.DataFrame(columns=['.geo', f'rh98_{year_4}', f'rh75_{year_4}', f'rh50_{year_4}', f'ch_{year_4}'])

        try:
            df_3 = pd.read_csv(file_3)
            df_3.rename(columns={'rh98_class': f'rh98_{year_3}', 'rh75_class': f'rh75_{year_3}', 'rh50_class': f'rh50_{year_3}'}, inplace=True)

        except Exception as e:
            print(e)
            df_3 = pd.DataFrame(columns=['.geo', f'rh98_{year_3}', f'rh75_{year_3}', f'rh50_{year_3}', f'ch_{year_3}'])

        merged_df = pd.merge(df_4, df_3, on='.geo', how='outer')
        del(df_4)
        del(df_3)

        try:
            df_2 = pd.read_csv(file_2)
            df_2.rename(columns={'rh98_class': f'rh98_{year_2}', 'rh75_class': f'rh75_{year_2}', 'rh50_class': f'rh50_{year_2}'}, inplace=True)

        except Exception as e:
            print(e)
            df_2 = pd.DataFrame(columns=['.geo', f'rh98_{year_2}', f'rh75_{year_2}', f'rh50_{year_2}', f'ch_{year_2}'])

        merged_df = pd.merge(merged_df, df_2, on='.geo', how='outer')
        del(df_2)

        try:
            df_1 = pd.read_csv(file_1)
            df_1.rename(columns={'rh98_class': f'rh98_{year_1}', 'rh75_class': f'rh75_{year_1}', 'rh50_class': f'rh50_{year_1}'}, inplace=True)
        except Exception as e:
            print(e)
            df_1 = pd.DataFrame(columns=['.geo', f'rh98_{year_1}', f'rh75_{year_1}', f'rh50_{year_1}', f'ch_{year_1}'])

        merged_df = pd.merge(merged_df, df_1, on='.geo', how='outer')
        del(df_1)

        try:
            df_0 = pd.read_csv(file_0)
            df_0.rename(columns={'rh98_class': f'rh98_{year}', 'rh75_class': f'rh75_{year}', 'rh50_class': f'rh50_{year}'}, inplace=True)

        except Exception as e:
            print(e)
            df_0 = pd.DataFrame(columns=['.geo', f'rh98_{year}', f'rh75_{year}', f'rh50_{year}', f'ch_{year}'])

        merged_df = pd.merge(merged_df, df_0, on='.geo', how='outer')
        del(df_0)
        print("merged_df>>>",merged_df)

        try:
          merged_df = merged_df[['.geo', f'rh98_{year_4}', f'rh98_{year_3}', f'rh98_{year_2}', f'rh98_{year_1}', f'rh98_{year}',
                               f'rh75_{year_4}', f'rh75_{year_3}', f'rh75_{year_2}', f'rh75_{year_1}', f'rh75_{year}', f'rh50_{year_4}',
                               f'rh50_{year_3}', f'rh50_{year_2}', f'rh50_{year_1}', f'rh50_{year}']]
        except Exception as e:
          print(e)

        total_length += len(merged_df)
        print("Length of merged_df:", len(merged_df))
        print("Total Length:", total_length)
        if year == '2019':
          correction_df = ch_corrections_2017(merged_df) # Uncomment when correcting 2017
        else:
          correction_df = ch_corrections(merged_df) # Comment when correcting 2017

        del(merged_df)

        total_corrections += len(correction_df)
        print(f'Correction_df length: {len(correction_df)}')
        print(f'Total Corrections: {total_corrections}')
        if len(correction_df) > 0:

            choices = [0, 0, 1, 2, 1, 2]

            conditions = [
                (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 0),
                (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 1),
                (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 1) & (correction_df[f'rh98_{year_4}'] == 0),
                (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 1) & (correction_df[f'rh98_{year_4}'] == 1),
                (correction_df[f'rh50_{year_4}'] == 1) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 0),
                (correction_df[f'rh50_{year_4}'] == 1) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 1)
            ]
            correction_df[f'ch_{year_4}'] = np.select(conditions, choices, default=3)


            conditions = [
                (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 0),
                (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 1),
                (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 1) & (correction_df[f'rh98_{year_3}'] == 0),
                (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 1) & (correction_df[f'rh98_{year_3}'] == 1),
                (correction_df[f'rh50_{year_3}'] == 1) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 0),
                (correction_df[f'rh50_{year_3}'] == 1) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 1)
            ]
            correction_df[f'ch_{year_3}'] = np.select(conditions, choices, default=3)


            conditions = [
                (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 0),
                (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 1),
                (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 1) & (correction_df[f'rh98_{year_2}'] == 0),
                (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 1) & (correction_df[f'rh98_{year_2}'] == 1),
                (correction_df[f'rh50_{year_2}'] == 1) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 0),
                (correction_df[f'rh50_{year_2}'] == 1) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 1)
            ]
            correction_df[f'ch_{year_2}'] = np.select(conditions, choices, default=3)


            conditions = [
                (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 0),
                (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 1),
                (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 1) & (correction_df[f'rh98_{year_1}'] == 0),
                (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 1) & (correction_df[f'rh98_{year_1}'] == 1),
                (correction_df[f'rh50_{year_1}'] == 1) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 0),
                (correction_df[f'rh50_{year_1}'] == 1) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 1)
            ]
            correction_df[f'ch_{year_1}'] = np.select(conditions, choices, default=3)


            conditions = [
                (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 0),
                (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 1),
                (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 1) & (correction_df[f'rh98_{year}'] == 0),
                (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 1) & (correction_df[f'rh98_{year}'] == 1),
                (correction_df[f'rh50_{year}'] == 1) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 0),
                (correction_df[f'rh50_{year}'] == 1) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 1)
            ]
            correction_df[f'ch_{year}'] = np.select(conditions, choices, default=3)

            cols = correction_df.columns
            for j in range(1, len(cols)):
                correction_df[cols[j]] = correction_df[cols[j]].astype('Int64')
            # if year != last_year: # TODO remove this
            correction_year_2 = correction_df[['.geo', f'rh50_{year_2}', f'rh75_{year_2}', f'rh98_{year_2}', f'ch_{year_2}']]
            correction_year_2.to_csv(f'{path}{dist_list[i]}/{year_2}/result_chm_corrections.csv', index=False)
            if year != '2019':
              correction_year_1 = correction_df[['.geo', f'rh50_{year_1}', f'rh75_{year_1}', f'rh98_{year_1}', f'ch_{year_1}']] # Comment when correcting 2017
              correction_year_1.to_csv(f'{path}{dist_list[i]}/{year_1}/result_chm_corrections.csv', index=False) # Comment when correcting 2017

        del(correction_df)


Western Himalayan Region
len(dist_list): 4
dist_list=: ['Dehradun', 'Garhwal', 'Nainital', 'Champawat']
0 Dehradun
merged_df>>>                                                       .geo  rh98_2020  \
0        {"geodesic":false,"type":"Point","coordinates"...        0.0   
1        {"geodesic":false,"type":"Point","coordinates"...        0.0   
2        {"geodesic":false,"type":"Point","coordinates"...        0.0   
3        {"geodesic":false,"type":"Point","coordinates"...        0.0   
4        {"geodesic":false,"type":"Point","coordinates"...        0.0   
...                                                    ...        ...   
5126057  {"geodesic":false,"type":"Point","coordinates"...        0.0   
5126058  {"geodesic":false,"type":"Point","coordinates"...        0.0   
5126059  {"geodesic":false,"type":"Point","coordinates"...        0.0   
5126060  {"geodesic":false,"type":"Point","coordinates"...        0.0   
5126061  {"geodesic":false,"type":"Point","coordinates"...        0.0

/tmp/ipykernel_17578/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length: 1278553
Total Corrections: 1278553
1 Garhwal
merged_df>>>                                                       .geo  rh98_2020  \
0        {"geodesic":false,"type":"Point","coordinates"...        0.0   
1        {"geodesic":false,"type":"Point","coordinates"...        0.0   
2        {"geodesic":false,"type":"Point","coordinates"...        0.0   
3        {"geodesic":false,"type":"Point","coordinates"...        0.0   
4        {"geodesic":false,"type":"Point","coordinates"...        0.0   
...                                                    ...        ...   
9596162  {"geodesic":false,"type":"Point","coordinates"...        0.0   
9596163  {"geodesic":false,"type":"Point","coordinates"...        0.0   
9596164  {"geodesic":false,"type":"Point","coordinates"...        0.0   
9596165  {"geodesic":false,"type":"Point","coordinates"...        0.0   
9596166  {"geodesic":false,"type":"Point","coordinates"...        0.0   

         rh75_2020  rh50_2020  rh98_2021  r

/tmp/ipykernel_17578/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length: 1725432
Total Corrections: 3003985
2 Nainital
merged_df>>>                                                       .geo  rh98_2020  \
0        {"geodesic":false,"type":"Point","coordinates"...        0.0   
1        {"geodesic":false,"type":"Point","coordinates"...        0.0   
2        {"geodesic":false,"type":"Point","coordinates"...        0.0   
3        {"geodesic":false,"type":"Point","coordinates"...        0.0   
4        {"geodesic":false,"type":"Point","coordinates"...        0.0   
...                                                    ...        ...   
6912865  {"geodesic":false,"type":"Point","coordinates"...        1.0   
6912866  {"geodesic":false,"type":"Point","coordinates"...        1.0   
6912867  {"geodesic":false,"type":"Point","coordinates"...        1.0   
6912868  {"geodesic":false,"type":"Point","coordinates"...        1.0   
6912869  {"geodesic":false,"type":"Point","coordinates"...        1.0   

         rh75_2020  rh50_2020  rh98_2021  

/tmp/ipykernel_17578/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length: 1325096
Total Corrections: 4329081
3 Champawat
merged_df>>>                                                       .geo  rh98_2020  \
0        {"geodesic":false,"type":"Point","coordinates"...        0.0   
1        {"geodesic":false,"type":"Point","coordinates"...        1.0   
2        {"geodesic":false,"type":"Point","coordinates"...        1.0   
3        {"geodesic":false,"type":"Point","coordinates"...        0.0   
4        {"geodesic":false,"type":"Point","coordinates"...        1.0   
...                                                    ...        ...   
2951210  {"geodesic":false,"type":"Point","coordinates"...        0.0   
2951211  {"geodesic":false,"type":"Point","coordinates"...        0.0   
2951212  {"geodesic":false,"type":"Point","coordinates"...        0.0   
2951213  {"geodesic":false,"type":"Point","coordinates"...        0.0   
2951214  {"geodesic":false,"type":"Point","coordinates"...        1.0   

         rh75_2020  rh50_2020  rh98_2021 

/tmp/ipykernel_17578/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length: 762567
Total Corrections: 5091648


CH - correction Chunking (Run this only for those locations which are causing RAM issue)

In [ ]:
# Add imports at top of your file (if not already imported)
import sqlite3
import os
import math
from pathlib import Path

CHUNKSIZE = 4000_000
print(year)
for agroclimatic_zone in acz_list:
    print(agroclimatic_zone)

    df = pd.read_csv('drive/MyDrive/TreeHealth/district_to_agroclimaticZone_mapping.csv')
    df['IntersectingZones'] = df['IntersectingZones'].apply(ast.literal_eval)
    district_mapping_df = df[df['AgroclimaticZone'] == agroclimatic_zone][['District', 'IntersectingZones']]

    dist_list = district_mapping_df['District'].tolist()
    dist_list = ['Garhchiroli']
    print(f'len(dist_list): {len(dist_list)}')
    print(f'dist_list: {dist_list}')

    path = f'drive/MyDrive/TreeHealth/{agroclimatic_zone}/'
    total_corrections = 0
    total_length = 0


    for i in range(len(dist_list)):
      print(i, "District",dist_list[i])
      file_4 = f"{path}{dist_list[i]}/{year_4}/result_chm.csv"
      file_3 = f"{path}{dist_list[i]}/{year_3}/result_chm.csv"
      file_2 = f"{path}{dist_list[i]}/{year_2}/result_chm.csv"
      file_1 = f"{path}{dist_list[i]}/{year_1}/result_chm.csv"
      file_0 = f"{path}{dist_list[i]}/{year}/result_chm.csv"

      # --- Begin drop-in chunked-on-disk merge logic ---
      # Create temp sqlite DB per district to avoid collisions and keep things local to each district
      db_path = f'{path}{dist_list[i]}/temp_merge.db'
      # ensure directory exists
      Path(os.path.dirname(db_path)).mkdir(parents=True, exist_ok=True)
      # remove previous DB if exists (so headers won't collide when appending)
      if os.path.exists(db_path):
          try:
              os.remove(db_path)
          except Exception:
              pass

      conn = sqlite3.connect(db_path)
      # Use pragmas to speed inserts (safe for temp file)
      conn.execute('PRAGMA synchronous=OFF;')
      conn.execute('PRAGMA journal_mode=MEMORY;')
      conn.commit()

      # helper to stream a CSV into an sqlite table; normalizes column names: .geo -> geo, rh*_class -> rh*_<year>
      def stream_csv_to_table(csv_file, table_name, year_tag):
          # if file doesn't exist, create empty table with expected schema (so joins work)
          if not os.path.exists(csv_file)  or os.path.getsize(csv_file) <= 1:
              # create empty table with schema
              conn.execute(f'''
                  CREATE TABLE IF NOT EXISTS {table_name} (
                      geo TEXT PRIMARY KEY,
                      rh98_{year_tag} INTEGER,
                      rh75_{year_tag} INTEGER,
                      rh50_{year_tag} INTEGER
                  )
              ''')
              conn.commit()
              return

          cols = ['.geo', 'rh98_class', 'rh75_class', 'rh50_class']
          try:
              reader = pd.read_csv(csv_file, usecols=cols, chunksize=CHUNKSIZE, iterator=True)
          except Exception as e:
              # if reading with those columns fails, fall back to reading only .geo
              reader = pd.read_csv(csv_file, chunksize=CHUNKSIZE, iterator=True)

          any_rows = False

          for chunk in reader:
              any_rows = True
              # normalize column names and keep only what we need
              if '.geo' in chunk.columns:
                  chunk = chunk.rename(columns={
                      '.geo': 'geo',
                      'rh98_class': f'rh98_{year_tag}',
                      'rh75_class': f'rh75_{year_tag}',
                      'rh50_class': f'rh50_{year_tag}'
                  })
                  # ensure columns exist even if file lacked them
                  for c in [f'rh98_{year_tag}', f'rh75_{year_tag}', f'rh50_{year_tag}']:
                      if c not in chunk.columns:
                          chunk[c] = pd.NA

                  chunk[['geo', f'rh98_{year_tag}', f'rh75_{year_tag}', f'rh50_{year_tag}']].to_sql(
                      table_name, conn, if_exists='append', index=False
                  )

          # If the file had headers but no data rows,
          # make sure the table still exists with the right schema
          if not any_rows:
              conn.execute(f'''
                  CREATE TABLE IF NOT EXISTS {table_name} (
                      geo TEXT PRIMARY KEY,
                      rh98_{year_tag} INTEGER,
                      rh75_{year_tag} INTEGER,
                      rh50_{year_tag} INTEGER
                  )
              ''')
              conn.commit()

      # Stream each file into its own sqlite table
      print("File 4")
      stream_csv_to_table(file_4, 't4', year_4)
      print("File 3")
      stream_csv_to_table(file_3, 't3', year_3)
      print("File 2")
      stream_csv_to_table(file_2, 't2', year_2)
      print("File 1")
      stream_csv_to_table(file_1, 't1', year_1)
      print("File 0")
      stream_csv_to_table(file_0, 't0', year)

      # Build union of all geo keys (on-disk), then count total rows
      union_sql = '''
          SELECT geo FROM t4
          UNION
          SELECT geo FROM t3
          UNION
          SELECT geo FROM t2
          UNION
          SELECT geo FROM t1
          UNION
          SELECT geo FROM t0
      '''
      count_sql = f"SELECT COUNT(*) FROM ({union_sql}) AS u"
      cursor = conn.execute(count_sql)
      total_rows = cursor.fetchone()[0]
      print("Total unique .geo keys (union):", total_rows)

      # process in CHUNKSIZE slices from the union
      num_chunks = math.ceil(total_rows / CHUNKSIZE)
      print(f"Processing {num_chunks} merge-chunks of size up to {CHUNKSIZE}...")

      # build a parameterized select that left-joins each year's table to the union subquery
      chunk_select_template = f'''
          SELECT u.geo AS ".geo",
                t4.rh98_{year_4} as rh98_{year_4}, t4.rh75_{year_4} as rh75_{year_4}, t4.rh50_{year_4} as rh50_{year_4},
                t3.rh98_{year_3} as rh98_{year_3}, t3.rh75_{year_3} as rh75_{year_3}, t3.rh50_{year_3} as rh50_{year_3},
                t2.rh98_{year_2} as rh98_{year_2}, t2.rh75_{year_2} as rh75_{year_2}, t2.rh50_{year_2} as rh50_{year_2},
                t1.rh98_{year_1} as rh98_{year_1}, t1.rh75_{year_1} as rh75_{year_1}, t1.rh50_{year_1} as rh50_{year_1},
                t0.rh98_{year}   as rh98_{year},   t0.rh75_{year}   as rh75_{year},   t0.rh50_{year}   as rh50_{year}
          FROM (
              {union_sql}
              ORDER BY geo
              LIMIT ? OFFSET ?
          ) u
          LEFT JOIN t4 ON u.geo = t4.geo
          LEFT JOIN t3 ON u.geo = t3.geo
          LEFT JOIN t2 ON u.geo = t2.geo
          LEFT JOIN t1 ON u.geo = t1.geo
          LEFT JOIN t0 ON u.geo = t0.geo
      '''

      # remove any existing output correction files for this district/year so we can append anew
      out_file_y2 = f'{path}{dist_list[i]}/{year_2}/result_chm_corrections.csv'
      out_file_y1 = f'{path}{dist_list[i]}/{year_1}/result_chm_corrections.csv'
      # We'll append chunk outputs; delete existing so headers are added correctly (mimic original behaviour)
      if os.path.exists(out_file_y2):
          os.remove(out_file_y2)
      if os.path.exists(out_file_y1) and year != '2019':
          os.remove(out_file_y1)

      for chunk_idx in range(num_chunks):
          offset = chunk_idx * CHUNKSIZE
          params = (CHUNKSIZE, offset)
          # read chunk into pandas
          df_chunk = pd.read_sql_query(chunk_select_template, conn, params=params)

          # rename the 'year' columns to include the actual year variable names used in the template above
          # (they already are named properly except the generic 'rh98_{year}' placeholders which the SQL aliased as rh98_{year})
          # ensure types and presence of columns, then proceed exactly as your old code expects (select columns etc.)
          print(f"Processing SQL chunk {chunk_idx+1}/{num_chunks} with {len(df_chunk)} rows")
          total_length += len(df_chunk)
          print("Length of merged_df (this chunk):", len(df_chunk))
          print("Total Length so far:", total_length)

          # Recreate expected column selection (same as original)
          try:
              merged_df = df_chunk[['.geo',
                                    f'rh98_{year_4}', f'rh98_{year_3}', f'rh98_{year_2}', f'rh98_{year_1}', f'rh98_{year}',
                                    f'rh75_{year_4}', f'rh75_{year_3}', f'rh75_{year_2}', f'rh75_{year_1}', f'rh75_{year}',
                                    f'rh50_{year_4}', f'rh50_{year_3}', f'rh50_{year_2}', f'rh50_{year_1}', f'rh50_{year}']]
          except Exception as e:
              print("Column selection error on chunk:", e)
              # Continue with whatever columns are present
              merged_df = df_chunk.copy()

          # run correction on this chunk exactly like before
          if year == '2019':
              correction_df = ch_corrections_2017(merged_df)
          else:
              correction_df = ch_corrections(merged_df)

          del(merged_df)

          total_corrections += len(correction_df)
          print(f'Correction_df length (chunk): {len(correction_df)}')
          print(f'Total Corrections so far: {total_corrections}')

          if len(correction_df) > 0:
              # same CH assignment logic as before - keep exactly the same code
              choices = [0, 0, 1, 2, 1, 2]

              conditions = [
                  (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 0),
                  (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 1),
                  (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 1) & (correction_df[f'rh98_{year_4}'] == 0),
                  (correction_df[f'rh50_{year_4}'] == 0) & (correction_df[f'rh75_{year_4}'] == 1) & (correction_df[f'rh98_{year_4}'] == 1),
                  (correction_df[f'rh50_{year_4}'] == 1) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 0),
                  (correction_df[f'rh50_{year_4}'] == 1) & (correction_df[f'rh75_{year_4}'] == 0) & (correction_df[f'rh98_{year_4}'] == 1)
              ]
              correction_df[f'ch_{year_4}'] = np.select(conditions, choices, default=3)

              conditions = [
                  (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 0),
                  (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 1),
                  (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 1) & (correction_df[f'rh98_{year_3}'] == 0),
                  (correction_df[f'rh50_{year_3}'] == 0) & (correction_df[f'rh75_{year_3}'] == 1) & (correction_df[f'rh98_{year_3}'] == 1),
                  (correction_df[f'rh50_{year_3}'] == 1) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 0),
                  (correction_df[f'rh50_{year_3}'] == 1) & (correction_df[f'rh75_{year_3}'] == 0) & (correction_df[f'rh98_{year_3}'] == 1)
              ]
              correction_df[f'ch_{year_3}'] = np.select(conditions, choices, default=3)

              conditions = [
                  (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 0),
                  (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 1),
                  (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 1) & (correction_df[f'rh98_{year_2}'] == 0),
                  (correction_df[f'rh50_{year_2}'] == 0) & (correction_df[f'rh75_{year_2}'] == 1) & (correction_df[f'rh98_{year_2}'] == 1),
                  (correction_df[f'rh50_{year_2}'] == 1) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 0),
                  (correction_df[f'rh50_{year_2}'] == 1) & (correction_df[f'rh75_{year_2}'] == 0) & (correction_df[f'rh98_{year_2}'] == 1)
              ]
              correction_df[f'ch_{year_2}'] = np.select(conditions, choices, default=3)

              conditions = [
                  (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 0),
                  (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 1),
                  (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 1) & (correction_df[f'rh98_{year_1}'] == 0),
                  (correction_df[f'rh50_{year_1}'] == 0) & (correction_df[f'rh75_{year_1}'] == 1) & (correction_df[f'rh98_{year_1}'] == 1),
                  (correction_df[f'rh50_{year_1}'] == 1) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 0),
                  (correction_df[f'rh50_{year_1}'] == 1) & (correction_df[f'rh75_{year_1}'] == 0) & (correction_df[f'rh98_{year_1}'] == 1)
              ]
              correction_df[f'ch_{year_1}'] = np.select(conditions, choices, default=3)

              conditions = [
                  (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 0),
                  (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 1),
                  (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 1) & (correction_df[f'rh98_{year}'] == 0),
                  (correction_df[f'rh50_{year}'] == 0) & (correction_df[f'rh75_{year}'] == 1) & (correction_df[f'rh98_{year}'] == 1),
                  (correction_df[f'rh50_{year}'] == 1) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 0),
                  (correction_df[f'rh50_{year}'] == 1) & (correction_df[f'rh75_{year}'] == 0) & (correction_df[f'rh98_{year}'] == 1)
              ]
              correction_df[f'ch_{year}'] = np.select(conditions, choices, default=3)

              cols = correction_df.columns
              for j in range(1, len(cols)):
                  correction_df[cols[j]] = correction_df[cols[j]].astype('Int64')

              # Write outputs appending chunk results (same filenames as before)
              # if year != last_year:
              # correction_year_2 = correction_df[['.geo', f'rh50_{year_2}', f'rh75_{year_2}', f'rh98_{year_2}', f'ch_{year_2}']]
              correction_year_2 = correction_df[correction_df[f'ch_{year_2}'].notna()][['.geo', f'rh50_{year_2}', f'rh75_{year_2}', f'rh98_{year_2}', f'ch_{year_2}']]
              correction_year_2.to_csv(out_file_y2, mode='a', index=False, header=not os.path.exists(out_file_y2))

              if year != '2019':
                  # correction_year_1 = correction_df[['.geo', f'rh50_{year_1}', f'rh75_{year_1}', f'rh98_{year_1}', f'ch_{year_1}']]
                  correction_year_1 = correction_df[correction_df[f'ch_{year_1}'].notna()][['.geo', f'rh50_{year_1}', f'rh75_{year_1}', f'rh98_{year_1}', f'ch_{year_1}']]
                  correction_year_1.to_csv(out_file_y1, mode='a', index=False, header=not os.path.exists(out_file_y1))

          del(correction_df)

      # cleanup sqlite DB
      conn.close()
      try:
          os.remove(db_path)
      except Exception:
          pass

      # --- End drop-in chunked-on-disk merge logic ---

2024
Eastern Plateau & Hills Region
len(dist_list): 1
dist_list: ['Garhchiroli']
0 District Garhchiroli
File 4
File 3
File 2
File 1
File 0
Total unique .geo keys (union): 20257293
Processing 6 merge-chunks of size up to 4000000...
Processing SQL chunk 1/6 with 4000000 rows
Length of merged_df (this chunk): 4000000
Total Length so far: 4000000


/tmp/ipykernel_6773/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length (chunk): 251390
Total Corrections so far: 251390
Processing SQL chunk 2/6 with 4000000 rows
Length of merged_df (this chunk): 4000000
Total Length so far: 8000000


/tmp/ipykernel_6773/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length (chunk): 365821
Total Corrections so far: 617211
Processing SQL chunk 3/6 with 4000000 rows
Length of merged_df (this chunk): 4000000
Total Length so far: 12000000


...

Correction_df length (chunk): 626236
Total Corrections so far: 1773602
Processing SQL chunk 5/6 with 4000000 rows
Length of merged_df (this chunk): 4000000
Total Length so far: 20000000


/tmp/ipykernel_6773/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length (chunk): 824080
Total Corrections so far: 2597682
Processing SQL chunk 6/6 with 257293 rows
Length of merged_df (this chunk): 257293
Total Length so far: 20257293


/tmp/ipykernel_6773/833223507.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  correction_df.drop_duplicates(inplace=True)


Correction_df length (chunk): 54985
Total Corrections so far: 2652667
